# LAB-5: Sistema Avançado com Mapa do Ambiente e Pathfinding

Neste laboratório, implementaremos:
- **Mapa de grade do ambiente** construído em tempo real
- **Algoritmo A*** para pathfinding ótimo
- **Mapa de calor** mostrando ocupação dos obstáculos
- **Planejamento de trajetória** global

In [ ]:
# CÓDIGO LAB-5: Sistema Avançado com Mapa e Pathfinding
import pygame
import math
import numpy as np
from collections import deque, defaultdict
import heapq

LARGURA, ALTURA = 1000, 800
FPS = 60
COR_FUNDO = (20, 24, 30)
COR_ROBO = (0, 200, 255)
COR_OBSTACULO = (180, 50, 50)
COR_RAIO_LIVRE = (0, 255, 100)
COR_RAIO_COLISAO = (255, 200, 0)
COR_ALVO = (255, 100, 100)
COR_CAMINHO = (100, 255, 100)

class OccupancyGrid:
    """Mapa de grade de ocupação do ambiente."""
    
    def __init__(self, width, height, cell_size=20):
        self.cell_size = cell_size
        self.width = width
        self.height = height
        self.cols = width // cell_size
        self.rows = height // cell_size
        self.grid = np.zeros((self.rows, self.cols), dtype=np.float32)
        self.obstacle_threshold = 0.5

    def world_to_grid(self, x, y):
        """Converte coordenadas mundiais para grid."""
        col = int(x / self.cell_size)
        row = int(y / self.cell_size)
        return row, col

    def grid_to_world(self, row, col):
        """Converte coordenadas grid para mundiais."""
        x = col * self.cell_size + self.cell_size / 2
        y = row * self.cell_size + self.cell_size / 2
        return x, y

    def add_obstacle(self, obs):
        """Marca células como ocupadas baseado em obstáculo."""
        x, y, w, h = obs
        r1, c1 = self.world_to_grid(x, y)
        r2, c2 = self.world_to_grid(x + w, y + h)
        
        r1 = max(0, r1)
        r2 = min(self.rows, r2 + 1)
        c1 = max(0, c1)
        c2 = min(self.cols, c2 + 1)
        
        self.grid[r1:r2, c1:c2] = 1.0

    def is_free(self, row, col):
        """Verifica se uma célula está livre."""
        if row < 0 or row >= self.rows or col < 0 or col >= self.cols:
            return False
        return self.grid[row, col] < self.obstacle_threshold

    def draw(self, screen):
        """Desenha o mapa de calor."""
        for row in range(self.rows):
            for col in range(self.cols):
                value = self.grid[row, col]
                if value > 0:
                    # Escala de cor: preto para livre, vermelho para ocupado
                    intensity = int(value * 255)
                    cor = (intensity, 0, 0)
                    x = col * self.cell_size
                    y = row * self.cell_size
                    pygame.draw.rect(screen, cor, (x, y, self.cell_size, self.cell_size))


class PathPlanner:
    """Planejador de caminho usando A*."""
    
    def __init__(self, grid):
        self.grid = grid

    def heuristic(self, pos1, pos2):
        """Distância euclidiana como heurística."""
        return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)

    def get_neighbors(self, row, col):
        """Retorna vizinhos válidos (8-conectividade)."""
        neighbors = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = row + dr, col + dc
                if self.grid.is_free(nr, nc):
                    neighbors.append((nr, nc))
        return neighbors

    def plan(self, start, goal):
        """Planeja caminho usando A*."""
        start_grid = self.grid.world_to_grid(start[0], start[1])
        goal_grid = self.grid.world_to_grid(goal[0], goal[1])
        
        if not self.grid.is_free(*goal_grid):
            return []
        
        open_set = []
        heapq.heappush(open_set, (0, start_grid))
        came_from = {}
        g_score = defaultdict(lambda: float('inf'))
        g_score[start_grid] = 0
        
        closed_set = set()
        
        while open_set:
            _, current = heapq.heappop(open_set)
            
            if current == goal_grid:
                # Reconstrói caminho
                path = []
                while current in came_from:
                    x, y = self.grid.grid_to_world(current[0], current[1])
                    path.append((x, y))
                    current = came_from[current]
                return list(reversed(path))
            
            closed_set.add(current)
            
            for neighbor in self.get_neighbors(current[0], current[1]):
                if neighbor in closed_set:
                    continue
                
                # Calcula custo
                if neighbor[0] != current[0] and neighbor[1] != current[1]:
                    cost = 1.414  # diagonal
                else:
                    cost = 1.0  # ortogonal
                
                tentative_g = g_score[current] + cost
                
                if tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score = tentative_g + self.heuristic(neighbor, goal_grid)
                    heapq.heappush(open_set, (f_score, neighbor))
        
        return []


class AdvancedAutonomousRobot:
    """Robô com navegação baseada em mapa global."""
    
    def __init__(self, x, y, theta=0.0):
        self.x = float(x)
        self.y = float(y)
        self.theta = float(theta)
        
        # Sensores
        self.num_sensores = 8
        self.sensor_angles = [2 * math.pi * i / self.num_sensores for i in range(self.num_sensores)]
        self.sensor_range = 150.0
        self.sensor_readings = [self.sensor_range] * self.num_sensores
        
        # Navegação
        self.target_x = None
        self.target_y = None
        self.path = []
        self.path_index = 0
        self.velocidade = 0.0
        self.velocidade_max = 3.0
        self.aceleracao = 0.1
        self.trajetoria = deque(maxlen=500)

    def cast_rays(self, obstacles):
        """Verifica interseção dos raios."""
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            dx = math.cos(angle)
            dy = math.sin(angle)
            
            min_distance = self.sensor_range
            
            for obs in obstacles:
                distance = self._ray_rect_intersection(self.x, self.y, dx, dy, obs)
                if distance is not None and distance < min_distance:
                    min_distance = distance
            
            self.sensor_readings.append(min_distance)

    def _ray_rect_intersection(self, x0, y0, dx, dy, rect):
        """Calcula interseção ray-rectangle."""
        rx, ry, rw, rh = rect
        x_min, x_max = rx, rx + rw
        y_min, y_max = ry, ry + rh
        
        t_min = float('-inf')
        t_max = float('inf')
        
        if abs(dx) > 1e-6:
            t1 = (x_min - x0) / dx
            t2 = (x_max - x0) / dx
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif x0 < x_min or x0 > x_max:
            return None
        
        if abs(dy) > 1e-6:
            t1 = (y_min - y0) / dy
            t2 = (y_max - y0) / dy
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif y0 < y_min or y0 > y_max:
            return None
        
        if t_min < t_max and t_min > 0 and t_min < self.sensor_range:
            return t_min
        
        return None

    def set_target(self, x, y, path_planner):
        """Define alvo e planeja caminho."""
        self.target_x = x
        self.target_y = y
        self.path = path_planner.plan((self.x, self.y), (x, y))
        self.path_index = 0

    def update(self, obstacles):
        """Atualiza posição seguindo caminho planejado."""
        self.cast_rays(obstacles)
        
        min_sensor = min(self.sensor_readings)
        
        # Segue o caminho planejado
        if self.path and self.path_index < len(self.path):
            waypoint = self.path[self.path_index]
            
            dx = waypoint[0] - self.x
            dy = waypoint[1] - self.y
            dist = math.sqrt(dx**2 + dy**2)
            
            if dist < 15:  # Próximo do waypoint
                self.path_index += 1
            else:
                # Calcula ângulo desejado
                angle_desejado = math.atan2(dy, dx)
                delta_theta = angle_desejado - self.theta
                
                while delta_theta > math.pi:
                    delta_theta -= 2 * math.pi
                while delta_theta < -math.pi:
                    delta_theta += 2 * math.pi
                
                self.theta += 0.05 * delta_theta
                
                # Controle de velocidade
                if min_sensor < 50:
                    self.velocidade = max(0, self.velocidade - self.aceleracao)
                else:
                    self.velocidade = min(self.velocidade_max, self.velocidade + self.aceleracao)
        else:
            self.velocidade = 0
        
        # Atualiza posição
        self.x += self.velocidade * math.cos(self.theta)
        self.y += self.velocidade * math.sin(self.theta)
        
        self.x = max(0, min(LARGURA, self.x))
        self.y = max(0, min(ALTURA, self.y))
        
        self.trajetoria.append((self.x, self.y))

    def draw(self, screen):
        """Desenha o robô."""
        # Trajetória
        if len(self.trajetoria) > 1:
            pygame.draw.lines(screen, (100, 150, 200), list(self.trajetoria), 1)
        
        # Caminho planejado
        if self.path and self.path_index < len(self.path):
            pygame.draw.lines(screen, COR_CAMINHO, self.path[self.path_index:], 2)
        
        # Corpo
        pygame.draw.circle(screen, COR_ROBO, (int(self.x), int(self.y)), 8)
        pygame.draw.line(screen, COR_ROBO, (self.x, self.y),
                         (self.x + 12 * math.cos(self.theta),
                          self.y + 12 * math.sin(self.theta)), 2)
        
        # Raios (4 principais)
        for i in [0, 2, 4, 6]:
            beta = self.sensor_angles[i]
            angle = self.theta + beta
            distance = self.sensor_readings[i]
            end_x = self.x + distance * math.cos(angle)
            end_y = self.y + distance * math.sin(angle)
            
            cor = COR_RAIO_COLISAO if distance < self.sensor_range - 0.1 else COR_RAIO_LIVRE
            pygame.draw.line(screen, cor, (self.x, self.y), (end_x, end_y), 1)


def draw_obstacles(screen, obstacles):
    """Desenha obstáculos."""
    for obs in obstacles:
        x, y, w, h = obs
        pygame.draw.rect(screen, COR_OBSTACULO, (x, y, w, h))


def draw_alvo(screen, x, y, raio=10):
    """Desenha o alvo."""
    if x is not None and y is not None:
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio)
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio - 2, 2)


def main():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB-5: Mapa do Ambiente e Pathfinding A*")
    clock = pygame.time.Clock()
    
    obstacles = [
        (100, 100, 150, 30),
        (400, 150, 30, 200),
        (650, 400, 150, 30),
        (300, 450, 250, 30),
        (50, 350, 30, 150),
        (700, 100, 50, 250),
    ]
    
    # Cria mapa de ocupação
    grid = OccupancyGrid(LARGURA, ALTURA, cell_size=20)
    for obs in obstacles:
        grid.add_obstacle(obs)
    
    # Cria planejador de caminho
    planner = PathPlanner(grid)
    
    # Cria robô
    robot = AdvancedAutonomousRobot(100, 100)
    robot.set_target(850, 650, planner)
    
    show_grid = True
    running = True
    
    while running:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_g:
                    show_grid = not show_grid
            elif event.type == pygame.MOUSEBUTTONDOWN:
                x, y = pygame.mouse.get_pos()
                robot.set_target(x, y, planner)
        
        robot.update(obstacles)
        
        # Renderiza
        screen.fill(COR_FUNDO)
        
        # Mapa de ocupação (opcional)
        if show_grid:
            grid.draw(screen)
        
        draw_obstacles(screen, obstacles)
        robot.draw(screen)
        draw_alvo(screen, robot.target_x, robot.target_y)
        
        # HUD
        font = pygame.font.Font(None, 20)
        info_texts = [
            f"Posição: ({robot.x:.0f}, {robot.y:.0f})",
            f"Velocidade: {robot.velocidade:.2f}",
            f"Waypoint: {robot.path_index}/{len(robot.path)}",
            f"Caminho planejado: {len(robot.path)} waypoints",
        ]
        
        for i, text in enumerate(info_texts):
            surface = font.render(text, True, (200, 200, 200))
            screen.blit(surface, (10, 10 + i * 25))
        
        inst_font = pygame.font.Font(None, 16)
        inst = inst_font.render("Click: Novo alvo | G: Mapa | ESC: Sair", True, (150, 150, 150))
        screen.blit(inst, (10, ALTURA - 25))
        
        pygame.display.flip()
    
    pygame.quit()

if __name__ == "__main__":
    main()